<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.5-docker-and-containerization/notebooks/exercises-10_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.5: Containerization with Docker

*Module 10 · AI System Architecture & Production Deployment*

> "It works on my machine" ends here. Docker packages your code, dependencies, and runtime into one image that runs identically on your laptop, in CI, and on Cloud Run. This lesson builds production Dockerfiles for AI apps — from naive single-stage to optimized multi-stage builds.

---

## About this notebook

This notebook contains the **8 runnable code examples** from the published lesson `lesson-10.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The Basic Dockerfile — Works, But Bloated — `Dockerfile.basic`
2. Step 2 — Optimized Single-Stage — Slim Base + Layer Caching — `.dockerignore`
3. Step 2 — Optimized Single-Stage — Slim Base + Layer Caching — `Dockerfile.optimized`
4. Step 3 — Multi-Stage Build — Build Heavy, Ship Light — `Dockerfile.multistage`
5. Step 4 — Dependency Management — Reproducible Installs — `requirements.txt`
6. Step 4 — Dependency Management — Reproducible Installs — `Dockerfile.uv`
7. Step 5 — Cloud Run Deployment — The Complete Production Pipeline — `main.py`
8. Step 5 — Cloud Run Deployment — The Complete Production Pipeline — `deploy.sh`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q numpy langchain fastapi uvicorn redis pydantic


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The Basic Dockerfile — Works, But Bloated

The Dockerfile most tutorials teach. It works — but it's 1.2 GB and slow to build.

**`Dockerfile.basic`** · _Block 1 of 8_

❌ THE BASIC DOCKERFILE (don't use this) — Problems: huge image, slow builds, no caching


In [ ]:
# ❌ THE BASIC DOCKERFILE (don't use this)
# Problems: huge image, slow builds, no caching

FROM python:3.12          # ← 1 GB base image (includes gcc, dev tools)

WORKDIR /app

COPY . .                  # ← copies EVERYTHING (venv, __pycache__, .git)

RUN pip install -r requirements.txt   # ← reinstalls all deps on EVERY code change

CMD ["python", "main.py"]

# Result:
#   Image size:  ~1.2 GB
#   Build time:  3-5 minutes (reinstalls deps every time)
#   Cold start:  slow (big image to download)


### Step 2 · Optimized Single-Stage — Slim Base + Layer Caching

Three fixes: slim image, dependency caching, and .dockerignore. 400 MB → 60s builds.

**`.dockerignore`** · _Block 2 of 8_

.dockerignore — exclude from Docker build context


In [ ]:
# .dockerignore — exclude from Docker build context
__pycache__/
*.pyc
.venv/
venv/
.env
.git/
.gitignore
*.md
tests/
docs/
.mypy_cache/
.pytest_cache/
*.egg-info/
dist/
build/


**`Dockerfile.optimized`** · _Block 3 of 8_

✅ OPTIMIZED SINGLE-STAGE DOCKERFILE — Slim base, layer caching, non-root user.


In [ ]:
# ✅ OPTIMIZED SINGLE-STAGE DOCKERFILE
# Slim base, layer caching, non-root user.

# Fix 1: Use slim base (150 MB vs 1 GB)
FROM python:3.12-slim

# Fix 2: Don't run as root
RUN useradd --create-home appuser

WORKDIR /app

# Fix 3: Copy deps FIRST (layer caching!)
# This layer only rebuilds when requirements.txt changes
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Fix 4: Copy code SECOND (changes frequently)
# Only this layer rebuilds on code changes
COPY . .

# Switch to non-root user
USER appuser

# Cloud Run sets PORT env var
ENV PORT=8080
EXPOSE 8080

# Health check for Cloud Run
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8080/health')" || exit 1

CMD ["python", "main.py"]

# Result:
#   Image size:  ~400 MB
#   Build time:  30-60s (deps cached, only code layer rebuilds)
#   Security:    non-root user


> **What just happened?** Four fixes: (1) Slim base — python:3.12-slim is 150 MB instead of 1 GB. (2) Layer caching — copy requirements.txt first, install deps, THEN copy code. Code changes don't trigger dep reinstall. (3) .dockerignore — excludes .git, __pycache__, .venv, tests. (4) Non-root user — security best practice. Image: 400 MB. Rebuild on code change: ~10 seconds (only the COPY layer). This is good enough for most apps. But we can go further.


### Step 3 · Multi-Stage Build — Build Heavy, Ship Light

Stage 1 installs everything (including build tools). Stage 2 copies only what's needed to run.

**`Dockerfile.multistage`** · _Block 4 of 8_

═══════════════════════════════════════════ — MULTI-STAGE DOCKERFILE FOR AI APPS


In [ ]:
# ═══════════════════════════════════════════
# MULTI-STAGE DOCKERFILE FOR AI APPS
# Stage 1: Build (install deps with build tools)
# Stage 2: Run (slim image, only runtime files)
# ═══════════════════════════════════════════

# ── STAGE 1: BUILD ──
FROM python:3.12-slim AS builder

# Install build dependencies (needed for numpy, etc.)
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /build

# Create a virtual environment for clean isolation
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── STAGE 2: RUNTIME ──
FROM python:3.12-slim AS runtime

# Copy ONLY the virtual environment (no build tools!)
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Non-root user
RUN useradd --create-home appuser
WORKDIR /app

# Copy application code
COPY . .

USER appuser

ENV PORT=8080
ENV PYTHONUNBUFFERED=1
ENV PYTHONDONTWRITEBYTECODE=1

EXPOSE 8080

HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:${PORT}/health')" || exit 1

CMD ["python", "main.py"]

# Result:
#   Image size:  ~200 MB (build tools NOT in final image)
#   Security:    no gcc/make in runtime, non-root
#   Cold start:  faster (smaller download)


> **What just happened?** Two stages: Builder has gcc/make (needed to compile numpy, grpcio, etc.) and installs all deps into a venv. Runtime copies only the venv from the builder — no build tools, no apt cache, no pip cache. The final image is ~200 MB instead of 400 MB. Build tools (gcc, make, dev headers) are in the builder stage only and never ship to production. Smaller image = faster Cloud Run cold starts, faster deploys, smaller attack surface.


### Step 4 · Dependency Management — Reproducible Installs

Pin every version. Lock the dependency tree. Never get "works on my machine" again.

**`requirements.txt`** · _Block 5 of 8_

requirements.txt — PIN EVERY VERSION — Generate with: pip freeze > requirements.txt


In [ ]:
# requirements.txt — PIN EVERY VERSION
# Generate with: pip freeze > requirements.txt
# Or use: uv pip compile pyproject.toml -o requirements.txt

fastapi==0.115.6
uvicorn[standard]==0.34.0
google-generativeai==0.8.5
pydantic==2.10.5
numpy==2.2.2
redis==5.2.1
langchain-google-genai==2.0.9
mcp==1.2.0


**`Dockerfile.uv`** · _Block 6 of 8_

═══════════════════════════════════════════ — DOCKERFILE WITH UV (10-100x faster pip)


In [ ]:
# ═══════════════════════════════════════════
# DOCKERFILE WITH UV (10-100x faster pip)
# uv is a Rust-based Python package installer
# that resolves and installs 10-100x faster.
# ═══════════════════════════════════════════

# ── STAGE 1: BUILD with uv ──
FROM python:3.12-slim AS builder

# Install uv (10-100x faster than pip)
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

WORKDIR /build

# Create venv with uv
RUN uv venv /opt/venv
ENV VIRTUAL_ENV=/opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Install dependencies (blazing fast)
COPY requirements.txt .
RUN uv pip install --no-cache -r requirements.txt

# ── STAGE 2: RUNTIME ──
FROM python:3.12-slim AS runtime

COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

RUN useradd --create-home appuser
WORKDIR /app
COPY . .
USER appuser

ENV PORT=8080 PYTHONUNBUFFERED=1

HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8080/health')" || exit 1

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]

# Result:
#   Build time: 10-15s (uv) vs 60-90s (pip)
#   Image size: ~200 MB (same as multi-stage)
#   Reproducible: uv lock file pins exact versions


> **What just happened?** Two approaches: pip freeze pins exact versions in requirements.txt (simple, universal). uv (Rust-based installer) installs 10-100x faster than pip — critical for CI/CD where you build images 50 times a day. The uv Dockerfile copies the uv binary from its official image, creates a venv, and installs deps. Build time drops from 60-90s to 10-15s. For production: always pin versions. For speed: use uv. For compatibility: use pip.


### Step 5 · Cloud Run Deployment — The Complete Production Pipeline

Build → push → deploy → verify. One script to go from code to production.

**`main.py`** · _Block 7 of 8_

FASTAPI APP — Cloud Run ready — Health check, structured logging, graceful shutdown.


In [ ]:
# =============================================
# FASTAPI APP — Cloud Run ready
# Health check, structured logging, graceful shutdown.
# =============================================

from fastapi import FastAPI
import os, json, time
from datetime import datetime

app = FastAPI(title="Netsetos AI Service")

START_TIME = time.time()
VERSION = os.getenv("VERSION", "1.0.0")

# ── Health check (required for Cloud Run) ──
@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "version": VERSION,
        "uptime_s": round(time.time() - START_TIME),
    }

# ── Your AI endpoint ──
@app.post("/v1/chat")
async def chat(request: dict):
    # Your LLM logic here
    return {"response": "Hello from the containerized AI service!"}

# ── Startup logging ──
@app.on_event("startup")
async def startup():
    print(json.dumps({
        "severity": "INFO",
        "message": "Service started",
        "version": VERSION,
        "port": os.getenv("PORT", "8080"),
    }))

if __name__ == "__main__":
    import uvicorn
    port = int(os.getenv("PORT", "8080"))
    uvicorn.run(app, host="0.0.0.0", port=port)


**`deploy.sh`** · _Block 8 of 8_

!/bin/bash — ═══════════════════════════════════════════


In [ ]:
#!/bin/bash
# ═══════════════════════════════════════════
# COMPLETE CLOUD RUN DEPLOYMENT PIPELINE
# Build → Push → Deploy → Verify
# ═══════════════════════════════════════════

# ── Config ──
PROJECT_ID="your-project-id"
REGION="asia-south1"
SERVICE="netsetos-ai"
VERSION=$(git rev-parse --short HEAD 2>/dev/null || echo "latest")
IMAGE="gcr.io/${PROJECT_ID}/${SERVICE}:${VERSION}"

echo "🚀 Deploying ${SERVICE} v${VERSION} to ${REGION}"

# ── Option A: Build locally + push ──
echo "📦 Building container..."
docker build -t ${IMAGE} -f Dockerfile.multistage .

echo "⬆️  Pushing to Container Registry..."
docker push ${IMAGE}

echo "🌐 Deploying to Cloud Run..."
gcloud run deploy ${SERVICE} \
  --image ${IMAGE} \
  --project ${PROJECT_ID} \
  --region ${REGION} \
  --port 8080 \
  --memory 512Mi \
  --cpu 1 \
  --min-instances 0 \
  --max-instances 10 \
  --timeout 60 \
  --set-env-vars "VERSION=${VERSION}" \
  --set-secrets "GOOGLE_API_KEY=google-api-key:latest" \
  --no-allow-unauthenticated

# ── Option B: Build on Cloud Build (no local Docker needed) ──
# gcloud run deploy ${SERVICE} \
#   --source . \
#   --project ${PROJECT_ID} \
#   --region ${REGION}

# ── Verify ──
echo ""
echo "✅ Deployment complete!"
URL=$(gcloud run services describe ${SERVICE} \
  --project ${PROJECT_ID} --region ${REGION} \
  --format "value(status.url)")
echo "   URL: ${URL}"

# Health check
echo "🏥 Health check..."
TOKEN=$(gcloud auth print-identity-token)
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" \
  -H "Authorization: Bearer ${TOKEN}" \
  "${URL}/health")
echo "   Status: ${HTTP_CODE}"

# Image size
echo "📏 Image size:"
docker images ${IMAGE} --format "   {{.Size}}"


> **What just happened?** The complete pipeline: (1) Build locally using the multi-stage Dockerfile. (2) Push to Google Container Registry. (3) Deploy to Cloud Run with: 512 MB memory, 1 CPU, auto-scaling 0-10, secrets from Secret Manager (not env vars!), IAM auth. (4) Verify — health check the deployed service, print the URL and image size. Option B: --source . lets Cloud Build do everything without Docker installed locally. Git commit hash as the version tag means you can always trace a running container back to exact code.


---

## Wrap-up

You've walked through all 8 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
